In [0]:
#Ingesting the data from the csv of products into a dataframe
df_product_raw = spark.read.csv("/Volumes/final_project/bronze/data/Product.csv", header=True, inferSchema=True, sep="\t")
display(df_product_raw)

In [0]:
df_product_raw.printSchema()

In [0]:
#Changing the columns of df_products_raw to snake_case naming
df_product_clean = df_product_raw.withColumnRenamed("ProductKey", "product_key").withColumnRenamed("Product", "product").withColumnRenamed("Standard Cost", "standard_cost").withColumnRenamed("Color", "color").withColumnRenamed("Subcategory", "subcategory").withColumnRenamed("Category", "category").withColumnRenamed("Background Color Format", "background_color_format").withColumnRenamed("Font Color Format", "font_color_format")
display(df_product_clean)


In [0]:
#Removing the $ symbol from standard_cost and converting it to demical data type
from pyspark.sql.functions import col, regexp_replace, trim
df_product_clean = df_product_clean.withColumn("standard_cost",regexp_replace(col("standard_cost"), r"[\$,]", "").cast("decimal(10,2)"))
display(df_product_clean)

In [0]:
#See how many rows have duplicates
total_count = df_product_clean.count()
distinct_count = df_product_clean.distinct().count()

print(f"Total Rows: {total_count}")
print(f"Unique Rows: {distinct_count}")
print(f"Number of Duplicate Rows: {total_count - distinct_count}")

In [0]:
#Drop duplicate rows
df_product_clean = df_product_clean.dropDuplicates()
display(df_product_clean)


In [0]:
from pyspark.sql.functions import col, regexp_replace, trim, lower

# 1. Replace all dashes (-) and commas (,) with a single space
# 2. Reduce multiple spaces to one space
# 3. Trim the ends
df_product_clean = df_product_clean.withColumn(
    "product", 
    trim(regexp_replace(regexp_replace(col("product"), r"[- ,]+", " "), r"\s+", " "))
)

display(df_product_clean)

In [0]:
from pyspark.sql.functions import col, regexp_replace, trim, expr

# 1. Dynamically remove the value of the 'color' column from the 'product' string
# 2. Use regexp_replace to turn dashes and commas into spaces
# 3. Clean up double spaces left behind
df_product_clean = (df_product_clean
    .withColumn("product", expr("regexp_replace(product, color, '')"))
    .withColumn("product", regexp_replace(col("product"), r"[- ,]+", " "))
    .withColumn("product", trim(regexp_replace(col("product"), r"\s+", " "))))


display(df_product_clean)

In [0]:
#Ensure the silver shecma exists
spark.sql("CREATE SCHEMA IF NOT EXISTS final_project.silver")
#Save DataFrame in the silver schema as a delta table
df_product_clean.write.format("delta").mode("overwrite").saveAsTable("final_project.silver.product")


In [0]:
#Display table
display(spark.table("final_project.silver.product"))